<a href="https://www.kaggle.com/code/rodrigoramooos/modelacao-projeto-de-previsao-de-atrasos-em-voos?scriptVersionId=311334855" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Modelação do Projeto: Previsão de Atraso e Cancelamento de Voos

## Sumário
Este notebook atende à **Milestones 3**, mantendo a leitura dos dados **diretamente do Kaggle**.  
O objetivo desta versão é realizar a **fase de modelação** do projeto.

## 1. Importação de Bibliotecas e Configuração do Ambiente

In [1]:
# Bibliotecas para Modelação
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# Carregar dataset
df = pd.read_csv('/kaggle/input/datasets/rodrigoramooos/flight-data-processed/flight_data_processed.csv')

print('Shape:', df.shape)
df.head()

Shape: (1041151, 339)


,month,day_of_month,day_of_week,distance,is_long_flight,cancelled,origin_ABI,origin_ABQ,origin_ABR,origin_ABY,...,origin_USA,origin_VCT,origin_VEL,origin_VLD,origin_VPS,origin_WRG,origin_XNA,origin_XWA,origin_YAK,origin_YUM
0,1,1,1,-0.550093,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,-0.359439,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,-0.922965,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,-0.922965,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,-1.009013,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# Definir variáveis
X = df.drop(columns=['cancelled'])
y = df['cancelled']

print('X:', X.shape)
print('y:', y.shape)

X: (1041151, 338)
y: (1041151,)


In [4]:
# Ver distribuição da variável alvo
y.value_counts(normalize=True) * 100

cancelled
0    98.472364
1     1.527636
Name: proportion, dtype: float64

In [5]:
# Divisão treino/teste (estratificada)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)

Treino: (832920, 338)
Teste: (208231, 338)


In [6]:
# Confirmar distribuição
print('Treino (%):')
print(y_train.value_counts(normalize=True) * 100)

print('\nTeste (%):')
print(y_test.value_counts(normalize=True) * 100)

Treino (%):
cancelled
0    98.472362
1     1.527638
Name: proportion, dtype: float64

Teste (%):
cancelled
0    98.47237
1     1.52763
Name: proportion, dtype: float64


## 1.2. Definição de Métricas de Desempenho

Como o nosso objetivo é prever o cancelamento de voos (um problema de classificação supervisionada), e o nosso dataset é altamente desequilibrado (apenas ~2.2% de cancelamentos).


Selecionámos as seguintes métricas para avaliar o modelo:
* **F1-Score (Métrica Principal):** Garante o equilíbrio entre a Precisão e o Recall. É a melhor métrica global para datasets com classes desequilibradas.
* **Recall / Sensibilidade:** Foco principal do negócio. Queremos minimizar os Falsos Negativos (prever que o voo opera, mas ele é cancelado), pois este erro causa os maiores custos logísticos e transtornos aos passageiros.
* **AUC-ROC:** Utilizada para avaliar a estabilidade do modelo em distinguir entre as duas classes, independentemente do limiar de decisão.

In [7]:
# Importação das métricas de desempenho selecionadas para Classificação
from sklearn.metrics import (
    f1_score, 
    recall_score, 
    precision_score, 
    roc_auc_score, 
)

print("Métricas de avaliação importadas com sucesso! Prontos para o Baseline.")

Métricas de avaliação importadas com sucesso! Prontos para o Baseline.


## 2.1. Implementação do Modelo Baseline

Para estabelecer um referencial de desempenho mínimo, implementamos um modelo de baixa complexidade: a **Regressão Logística**. O objetivo nesta fase não é obter o modelo perfeito, mas sim criar uma "linha de base" (baseline) de métricas. Qualquer modelo complexo que testarmos na próxima fase terá obrigatoriamente de superar os resultados deste modelo simples.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

print("A iniciar o treino do Modelo Baseline (Regressão Logística)...")

# 1. Instanciar o modelo
# max_iter=1000 garante que o algoritmo tem tempo matemático para convergir num dataset tão grande
# random_state=42 garante reprodutibilidade (dá sempre o mesmo resultado)
baseline_model = LogisticRegression(max_iter=1000, random_state=42)

# 2. Treinar o modelo com os dados de treino
baseline_model.fit(X_train, y_train)

# 3. Fazer previsões usando os dados de teste (que o modelo nunca viu)
y_pred = baseline_model.predict(X_test)
y_pred_proba = baseline_model.predict_proba(X_test)[:, 1] # Probabilidades para a Curva ROC

# 4. Quantificação de Desempenho
print("\n--- MATRIZ DE CONFUSÃO ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(classification_report(y_test, y_pred))

# Calcular e imprimir o AUC-ROC
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f"\nAUC-ROC Score: {auc_roc:.4f}")

A iniciar o treino do Modelo Baseline (Regressão Logística)...

--- MATRIZ DE CONFUSÃO ---
[[205050      0]
 [  3181      0]]

--- RELATÓRIO DE CLASSIFICAÇÃO ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99    205050
           1       0.00      0.00      0.00      3181

    accuracy                           0.98    208231
   macro avg       0.49      0.50      0.50    208231
weighted avg       0.97      0.98      0.98    208231


AUC-ROC Score: 0.7546


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
